# Segmentasi Instance Objek Ruangan dengan YOLO-seg dan Mask R-CNN

Evaluasi CPMK 3 Visi Komputer. Notebook ini melatih tiga model segmentasi instance pada dataset **Panoptic Indoor Segmentation v5** dari Roboflow, lalu menguji ketiganya pada test set asli dan pada test set foto ruangan yang diambil dan dianotasi sendiri.

| Kode | Model | Keluarga |
|---|---|---|
| M1 | YOLOv8n-seg | one-stage, CNN, mask dari prototype |
| M2 | YOLO11n-seg | one-stage, CNN, mask dari prototype |
| M3 | Mask R-CNN ResNet-50 FPN | two-stage, CNN, mask 28×28 per RoI |

NRP berakhiran 1, sehingga variabel yang diselidiki adalah **ukuran input 320 / 640 / 960**. YOLO11n-seg dilatih ulang pada ketiga ukuran, sedangkan Mask R-CNN yang dilatih pada 640 diuji pada ketiga ukuran saat inferensi.

**Cara menjalankan di Kaggle**

1. Settings → Accelerator → **GPU T4 x2**, lalu Settings → Internet → **On**.
2. Add-ons → Secrets → tambahkan `ROBOFLOW_API_KEY` dan centang untuk notebook ini.
3. Isi `MY_WORKSPACE` dan `MY_PROJECT` di sel konfigurasi setelah test set sendiri selesai dianotasi. Selama masih kosong, bagian test set sendiri dilewati.
4. Save Version → **Save & Run All (Commit)**. Setiap model dibatasi `TIME_H` jam GPU, jadi seluruh training memakan kurang lebih 5–6 jam.
5. Untuk run berikutnya, tambahkan output versi sebelumnya lewat Add Input agar training tidak diulang.

## 1. Persiapan lingkungan

In [ ]:
!pip install -q -U ultralytics roboflow torchmetrics pycocotools pillow-heif
import os, sys, json, time, glob, shutil, random, math, platform, zipfile
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, torchvision, yaml
import matplotlib.pyplot as plt
from PIL import Image
import ultralytics, torchmetrics

print("Python", platform.python_version())
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("Ultralytics", ultralytics.__version__, "| torchmetrics", torchmetrics.__version__)
assert torch.cuda.is_available(), "GPU belum aktif. Settings → Accelerator → GPU T4 x2"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU", GPU_NAME, f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

In [ ]:
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
WORK = Path("/kaggle/working") if IN_KAGGLE else Path("work")
TMP  = Path("/kaggle/temp") if IN_KAGGLE else Path("tmp")
OUT, DATA = WORK/"hasil_cpmk3", TMP/"data"
RUNS, FIG = OUT/"runs", OUT/"gambar"
for d in (OUT, RUNS, FIG, DATA): d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Dataset latih dari dosen
RF_WORKSPACE, RF_PROJECT, RF_VERSION = "panopticindoor", "panoptic-indoor-segmentation", 5
# Test set buatan sendiri di Roboflow, lihat URL app.roboflow.com/<workspace>/<project>/<versi>
MY_WORKSPACE, MY_PROJECT, MY_VERSION = "vk-a0mta", "tes-ruangan-sendiri", 3
# Foto kondisi sulit diberi nama berawalan ini, misalnya sulit_01_redup.jpg
HARD_PREFIX = "sulit"

IMGSZ = 640                  # ukuran input utama untuk perbandingan model
SIZES_NRP = [320, 640, 960]  # NRP ...1 → variabel ukuran input
TIME_H = 1.0                 # batas waktu training per model dalam jam (tugas: maksimal 2)
EPOCHS_MAX = 300             # batas atas saja, yang menghentikan training adalah TIME_H
CONF_EVAL = 0.001            # ambang skor untuk menghitung mAP
CONF_VIS = 0.25              # ambang skor untuk visualisasi dan analisis kegagalan
VAL_MAX = 300                # jumlah citra validasi untuk memilih epoch terbaik Mask R-CNN
COLORS = {"M1": "#2a78d6", "M2": "#eb6834", "M3": "#1baf7a"}

def find_prev_outputs():
    # Output versi notebook sebelumnya yang ditambahkan lewat Add Input
    found = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.count(os.sep) > 7: dirs[:] = []; continue
        if "hasil_cpmk3" in dirs: found.append(Path(root)/"hasil_cpmk3"); dirs.remove("hasil_cpmk3")
    if not found:   # dataset berisi hasil_cpmk3.zip yang belum diekstrak Kaggle
        for zp in glob.glob("/kaggle/input/**/hasil_cpmk3.zip", recursive=True):
            dst = TMP/"prev"/str(abs(hash(zp)))
            with zipfile.ZipFile(zp) as z: z.extractall(dst)
            if (dst/"hasil_cpmk3").exists(): found.append(dst/"hasil_cpmk3")
    return found
PREV = find_prev_outputs() if IN_KAGGLE else []
print("Output sebelumnya:", [str(p) for p in PREV] or "tidak ada")
if IN_KAGGLE and not PREV:
    print("PERHATIAN: bobot run sebelumnya tidak ditemukan di /kaggle/input, sehingga semua model akan dilatih ulang (±3,5 jam).\n"
          "Untuk melewati training, tambahkan output notebook versi sebelumnya atau dataset berisi folder hasil_cpmk3 lewat Add Input.")

GPU_LOG = OUT/"waktu_gpu.json"
gpu_log = json.loads(GPU_LOG.read_text()) if GPU_LOG.exists() else {}
for p in PREV:
    if (p/"waktu_gpu.json").exists():
        for k, v in json.loads((p/"waktu_gpu.json").read_text()).items(): gpu_log.setdefault(k, v)
def log_gpu(name, seconds, **extra):
    gpu_log[name] = {"detik": round(seconds, 1), "jam": round(seconds/3600, 3), "gpu": GPU_NAME,
                     "selesai": time.strftime("%Y-%m-%d %H:%M:%S"), **extra}
    GPU_LOG.write_text(json.dumps(gpu_log, indent=2))
GPU_LOG.write_text(json.dumps(gpu_log, indent=2))

def roboflow_key():
    k = os.environ.get("ROBOFLOW_API_KEY")
    if not k and IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        k = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    assert k, "API key Roboflow belum ada. Kaggle: Add-ons → Secrets → ROBOFLOW_API_KEY"
    return k
print("Platform:", "Kaggle Notebook" if IN_KAGGLE else "lokal", "| output:", OUT)

## 2. Dataset Panoptic Indoor Segmentation v5

Dataset diunduh dalam format YOLOv8 segmentasi, yaitu satu file `.txt` per citra dengan satu baris per objek berisi `kelas x1 y1 x2 y2 …` (titik poligon ternormalisasi 0–1). Jika ekspor itu tidak tersedia, notebook memakai ekspor COCO segmentation dan mengonversinya ke format yang sama. Daftar kelas dibaca langsung dari `data.yaml`.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_key())
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def names_of(root):
    n = yaml.safe_load(open(Path(root)/"data.yaml"))["names"]
    return [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)

def items_of(root, split):
    d = Path(root)/split/"images"
    imgs = sorted(p for p in d.glob("*") if p.suffix.lower() in IMG_EXT) if d.exists() else []
    return [(p, p.parent.parent/"labels"/(p.stem + ".txt")) for p in imgs]

def coco_to_yolo(src, dst):
    # Cadangan bila ekspor YOLOv8 tidak tersedia. Objek yang terdiri dari beberapa poligon ditulis per poligon
    from pycocotools import mask as mask_utils
    src, dst, names = Path(src), Path(dst), None
    for split in ("train", "valid", "test"):
        js = src/split/"_annotations.coco.json"
        if not js.exists(): continue
        coco = json.load(open(js))
        cats = [c for c in coco["categories"] if c.get("supercategory") != "none"] or coco["categories"]
        cats = sorted(cats, key=lambda c: c["id"]); cid = {c["id"]: i for i, c in enumerate(cats)}
        names = [c["name"] for c in cats]
        for sub in ("images", "labels"): (dst/split/sub).mkdir(parents=True, exist_ok=True)
        anns = {}
        for a in coco["annotations"]: anns.setdefault(a["image_id"], []).append(a)
        for im in coco["images"]:
            W, H, fn = im["width"], im["height"], Path(im["file_name"])
            shutil.copy(src/split/fn, dst/split/"images"/fn.name)
            lines = []
            for a in anns.get(im["id"], []):
                if a["category_id"] not in cid: continue
                seg, polys = a.get("segmentation"), []
                if isinstance(seg, list):
                    polys = [np.array(s).reshape(-1, 2) for s in seg if len(s) >= 6]
                elif isinstance(seg, dict):
                    rle = mask_utils.frPyObjects(seg, H, W) if isinstance(seg["counts"], list) else seg
                    cs, _ = cv2.findContours(mask_utils.decode(rle), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    polys = [c.reshape(-1, 2) for c in cs if len(c) >= 3]
                for p in polys:
                    lines.append(f"{cid[a['category_id']]} " + " ".join(f"{v:.6f}" for v in (p / [W, H]).flatten()))
            (dst/split/"labels"/(fn.stem + ".txt")).write_text("\n".join(lines))
    (dst/"data.yaml").write_text(yaml.safe_dump({"names": names}, allow_unicode=True))

def rf_download(ws, proj, ver, dest):
    dest = Path(dest)
    if (dest/"data.yaml").exists() and any(dest.glob("*/labels/*.txt")): return dest
    v = rf.workspace(ws).project(proj).version(ver)
    try:
        v.download("yolov8", location=str(dest), overwrite=True)
        assert any(dest.glob("*/labels/*.txt")), "label kosong"
    except Exception as e:
        print("Ekspor YOLOv8 gagal:", e, "→ memakai ekspor COCO segmentation")
        coco_dir = Path(str(dest) + "_coco")
        v.download("coco-segmentation", location=str(coco_dir), overwrite=True)
        coco_to_yolo(coco_dir, dest)
    return dest

SRC = rf_download(RF_WORKSPACE, RF_PROJECT, RF_VERSION, DATA/"panoptic_indoor_v5")

# Project ini punya kelas hasil salah ketik (93 kelas di halaman Roboflow). Typo yang jelas digabung ke nama yang benar,
# typo yang maknanya tidak jelas dibuang, lalu kelas tanpa satu pun instance dihapus dari daftar
CLASS_FIX = {"ligjt": "light", "liht": "light", "llight": "light", "persob": "person", "spon": "spoon", "spoom": "spoon",
             "palte": "plate", "pap": "paper", "cuo": "cup", "cyo": "cup", "towl": "towel", "staris": "stairs",
             "stovv": "stove-merged", "scissor": "scissors", "windo": "window-other", "ruf": "rug-merged", "phone": "cell phone"}
CLASS_DROP = {"poo", "st\\", "tra", "wink"}

def clean_classes(root):
    marker = Path(root)/"kelas_bersih.json"
    if marker.exists(): return json.loads(marker.read_text())["names"]
    raw = names_of(root); lbls = list(Path(root).glob("*/labels/*.txt")); count = {}
    merged = {i: (None if CLASS_FIX.get(n, n) in CLASS_DROP or n in CLASS_DROP else CLASS_FIX.get(n, n)) for i, n in enumerate(raw)}
    for lp in lbls:
        for line in lp.read_text().splitlines():
            p = line.split()
            if p and merged.get(int(p[0])): count[merged[int(p[0])]] = count.get(merged[int(p[0])], 0) + 1
    names = sorted(count)
    new = {i: (names.index(m) if m in count else None) for i, m in merged.items()}
    for lp in lbls:
        rows = [" ".join([str(new[int(l.split()[0])])] + l.split()[1:]) for l in lp.read_text().splitlines()
                if l.split() and new.get(int(l.split()[0])) is not None]
        lp.write_text("\n".join(rows))
    log = pd.DataFrame([{"kelas_asli": n, "menjadi": merged[i] or "(dibuang)", "instance": count.get(merged[i], 0) if merged[i] else 0}
                        for i, n in enumerate(raw) if merged[i] != n])
    marker.write_text(json.dumps({"names": names, "perubahan": log.to_dict("records")}, indent=2))
    print(f"Kelas di data.yaml: {len(raw)} → setelah dibersihkan: {len(names)}")
    if len(log): display(log)
    return names

NAMES = clean_classes(SRC); NC = len(NAMES)
shutil.copy(SRC/"kelas_bersih.json", OUT/"kelas_bersih.json")
if not items_of(SRC, "test"):
    # Versi tanpa split test: separuh validasi dipindah menjadi test agar data uji tidak dipakai memilih model
    val = items_of(SRC, "valid"); random.Random(SEED).shuffle(val)
    for sub in ("images", "labels"): (SRC/"test"/sub).mkdir(parents=True, exist_ok=True)
    for ip, lp in val[: len(val)//2]:
        shutil.move(str(ip), str(SRC/"test"/"images"/ip.name))
        if lp.exists(): shutil.move(str(lp), str(SRC/"test"/"labels"/lp.name))
    print("Split test tidak ada, dibuat dari 50% validasi")
DATA_YAML = SRC/"data_abs.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path": str(SRC.resolve()), "train": "train/images", "val": "valid/images",
                                     "test": "test/images", "names": dict(enumerate(NAMES))}, sort_keys=False, allow_unicode=True))
print(f"{NC} kelas:", NAMES)
for s in ("train", "valid", "test"): print(f"{s:6s}: {len(items_of(SRC, s))} citra")

### 2.1 Test set buatan sendiri

Foto dianotasi di project Roboflow sendiri dengan nama kelas yang sama. Urutan kelas di project baru bisa berbeda, sehingga indeks kelas dipetakan ulang berdasarkan **nama**. Semua split dari project itu digabung menjadi satu test set.

In [ ]:
def norm(s): return s.strip().lower().replace("_", " ").replace("-", " ")
OWN, OWN_MAX = None, 1280
if MY_PROJECT:
    raw = rf_download(MY_WORKSPACE, MY_PROJECT, MY_VERSION, DATA/"test_sendiri_raw")
    own_names, lookup = names_of(raw), {norm(n): i for i, n in enumerate(NAMES)}
    remap = {j: lookup.get(norm(CLASS_FIX.get(n, n))) for j, n in enumerate(own_names)}
    missing = [own_names[j] for j, i in remap.items() if i is None]
    if missing: print("PERINGATAN, kelas ini tidak ada di dataset latih dan diabaikan:", missing)
    OWN = DATA/"test_sendiri"
    for sub in ("images", "labels"): (OWN/"test"/sub).mkdir(parents=True, exist_ok=True)
    for split in ("train", "valid", "test"):
        for ip, lp in items_of(raw, split):
            # Foto HP berukuran 5712×3213. Sisi terpanjang diperkecil ke OWN_MAX agar mask ukuran penuh muat di memori GPU;
            # label YOLO dinormalisasi 0–1 sehingga tidak perlu diubah
            im = cv2.imread(str(ip)); s = OWN_MAX / max(im.shape[:2])
            if s < 1: im = cv2.resize(im, (round(im.shape[1] * s), round(im.shape[0] * s)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(OWN/"test"/"images"/ip.name), im, [cv2.IMWRITE_JPEG_QUALITY, 95])
            rows = []
            for line in (lp.read_text().splitlines() if lp.exists() else []):
                p = line.split()
                if p and remap.get(int(p[0])) is not None: rows.append(" ".join([str(remap[int(p[0])])] + p[1:]))
            (OWN/"test"/"labels"/(ip.stem + ".txt")).write_text("\n".join(rows))
    n_hard = sum(p.name.lower().startswith(HARD_PREFIX) for p, _ in items_of(OWN, "test"))
    print(f"Test set sendiri: {len(items_of(OWN, 'test'))} citra, {n_hard} di antaranya kondisi sulit")
else:
    print("MY_PROJECT masih kosong, evaluasi test set sendiri dilewati")

TEST_ASLI = items_of(SRC, "test")
TEST_OWN = items_of(OWN, "test") if OWN else []
def own_group(p): return "sendiri_sulit" if p.name.lower().startswith(HARD_PREFIX) else "sendiri_normal"

# Foto sendiri hanya dianotasi untuk sebagian kelas. Evaluasi test set sendiri, dan pembandingnya di test set asli,
# hanya memakai kelas-kelas ini agar prediksi kelas lain yang memang tidak dilabeli tidak dihitung sebagai FP
OWN_KEEP = sorted({int(l.split()[0]) for _, lp in TEST_OWN for l in lp.read_text().splitlines() if l.split()})
if TEST_OWN: print(f"Kelas yang dianotasi di foto sendiri ({len(OWN_KEEP)}):", [NAMES[i] for i in OWN_KEEP])

### 2.2 Statistik dataset

Ukuran objek mengikuti batas COCO, yaitu kecil di bawah 32² piksel, sedang sampai 96² piksel, dan besar di atasnya.

In [ ]:
def read_label(lp, W, H):
    objs = []
    for line in (Path(lp).read_text().splitlines() if Path(lp).exists() else []):
        p = line.split()
        if len(p) < 5: continue
        c, v = int(float(p[0])), np.array(p[1:], dtype=float)
        if len(v) == 4:
            xc, yc, w, h = v
            poly = np.array([[xc-w/2, yc-h/2], [xc+w/2, yc-h/2], [xc+w/2, yc+h/2], [xc-w/2, yc+h/2]])
        else:
            poly = v[: len(v)//2*2].reshape(-1, 2)
        objs.append((c, poly * [W, H]))
    return objs

def poly_mask(poly, W, H):
    m = np.zeros((H, W), np.uint8); cv2.fillPoly(m, [np.round(poly).astype(np.int32)], 1); return m

def gt_of(ip, lp):
    img = cv2.imread(str(ip)); H, W = img.shape[:2]
    objs = read_label(lp, W, H)
    masks = np.stack([poly_mask(p, W, H) for _, p in objs]).astype(bool) if objs else np.zeros((0, H, W), bool)
    labels = np.array([c for c, _ in objs], dtype=np.int64)
    keep = masks.any(axis=(1, 2))  # aman juga untuk citra tanpa objek (0 mask)
    return img, masks[keep], labels[keep]

def size_cat(a): return "kecil" if a < 32**2 else ("sedang" if a < 96**2 else "besar")

def stats(items, tag):
    rows = []
    for ip, lp in items:
        W, H = Image.open(ip).size
        objs = read_label(lp, W, H)
        for c, poly in objs:
            x, y = poly[:, 0], poly[:, 1]
            a = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
            rows.append({"set": tag, "kelas": NAMES[c] if c < NC else str(c), "luas": a, "ukuran": size_cat(a), "img": ip.name, "resolusi": f"{W}x{H}"})
        if not objs: rows.append({"set": tag, "kelas": None, "luas": np.nan, "ukuran": None, "img": ip.name, "resolusi": f"{W}x{H}"})
    return pd.DataFrame(rows)

ST = pd.concat([stats(items_of(SRC, s), s) for s in ("train", "valid", "test")] + ([stats(TEST_OWN, "sendiri")] if TEST_OWN else []))
ST.to_csv(OUT/"statistik_objek.csv", index=False)
obj = ST.dropna(subset=["kelas"])
print("Jumlah citra per set:"); display(ST.groupby("set")["img"].nunique())
print("Objek per citra (rata-rata):"); display((obj.groupby("set").size() / ST.groupby("set")["img"].nunique()).round(2))
print("Resolusi terbanyak per set:"); display(ST.groupby("set")["resolusi"].agg(lambda s: s.value_counts().head(3).to_dict()))
print("Proporsi ukuran objek (%):"); display((pd.crosstab(obj["set"], obj["ukuran"], normalize="index") * 100).round(1))
JUMLAH_KELAS = pd.crosstab(obj["kelas"], obj["set"]).reindex(NAMES).fillna(0).astype(int)
display(JUMLAH_KELAS.sort_values("train", ascending=False))

fig, ax = plt.subplots(figsize=(8, max(3, 0.32 * NC)))
s = JUMLAH_KELAS["train"].sort_values()
ax.barh(s.index, s.values, color=COLORS["M1"], height=0.7)
for y, v in enumerate(s.values): ax.text(v, y, f" {v}", va="center", fontsize=8)
ax.set_title("Jumlah instance per kelas di data latih"); ax.set_xlabel("instance")
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.savefig(FIG/"distribusi_kelas.png", dpi=150); plt.show()

In [ ]:
def class_color(c):
    r, g, b = plt.matplotlib.colors.hsv_to_rgb([(c * 0.618034) % 1.0, 0.75, 0.95])
    return (int(b * 255), int(g * 255), int(r * 255))

def overlay(img, masks, labels, scores=None):
    out = img.copy()
    for i, (m, c) in enumerate(zip(masks, labels)):
        col = np.array(class_color(int(c)), np.uint8)
        out[m] = (0.45 * col + 0.55 * out[m]).astype(np.uint8)
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, cs, -1, tuple(int(v) for v in col), 2)
        ys, xs = np.nonzero(m)
        txt = NAMES[int(c)] + (f" {float(scores[i]):.2f}" if scores is not None else "")
        org = (int(xs.min()), max(14, int(ys.min()) + 14))
        cv2.putText(out, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(out, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (20, 20, 20), 1, cv2.LINE_AA)
    return out[..., ::-1]

def show_gt_grid(items, title, fname, n=6):
    pick = random.Random(SEED).sample(items, min(n, len(items)))
    fig, axs = plt.subplots(2, 3, figsize=(15, 9))
    for ax in axs.flat: ax.axis("off")
    for ax, (ip, lp) in zip(axs.flat, pick):
        img, m, l = gt_of(ip, lp); ax.imshow(overlay(img, m, l)); ax.set_title(f"{ip.name[:28]} ({len(l)} objek)", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(FIG/fname, dpi=120); plt.show()

show_gt_grid(items_of(SRC, "train"), "Contoh ground truth data latih", "contoh_gt_latih.png")
if TEST_OWN: show_gt_grid(TEST_OWN, "Contoh ground truth test set sendiri", "contoh_gt_sendiri.png")

## 3. Model, evaluasi terpadu, dan training

Ketiga model dievaluasi dengan kode yang sama. Prediksi setiap model diubah menjadi mask biner seukuran citra asli, lalu mAP mask dan mAP box dihitung dengan `torchmetrics` (backend `pycocotools`, aturan COCO). Dengan cara ini angka YOLO dan Mask R-CNN dapat dibandingkan langsung.

In [ ]:
from torchmetrics.detection import MeanAveragePrecision
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import masks_to_boxes
import torchvision.transforms.functional as TF
DEV = torch.device("cuda:0")

def build_mrcnn(imgsz=IMGSZ):
    # Bobot COCO, head kelas dan head mask diganti sesuai jumlah kelas dataset (+1 latar)
    m = maskrcnn_resnet50_fpn(weights="DEFAULT", min_size=imgsz, max_size=imgsz,
                              box_detections_per_img=100, box_score_thresh=CONF_EVAL)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NC + 1)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(m.roi_heads.mask_predictor.conv5_mask.in_channels, 256, NC + 1)
    return m

def empty_pred(H, W):
    return {"boxes": torch.zeros((0, 4)), "scores": torch.zeros(0), "labels": torch.zeros(0, dtype=torch.long),
            "masks": torch.zeros((0, H, W), dtype=torch.bool)}

class YoloSeg:
    def __init__(self, weights, imgsz):
        self.m, self.imgsz = YOLO(str(weights)), imgsz
    def n_params(self): return sum(p.numel() for p in self.m.model.parameters())
    def __call__(self, img, conf=CONF_EVAL, imgsz=None):
        H, W = img.shape[:2]
        r = self.m.predict(img, imgsz=imgsz or self.imgsz, conf=conf, iou=0.7, max_det=100,
                           retina_masks=True, verbose=False, device=0)[0]
        if r.masks is None or len(r.boxes) == 0: return empty_pred(H, W)
        return {"boxes": r.boxes.xyxy.float().cpu(), "scores": r.boxes.conf.float().cpu(),
                "labels": r.boxes.cls.long().cpu(), "masks": r.masks.data.bool().cpu()}

class MaskRCNNSeg:
    def __init__(self, weights=None, imgsz=IMGSZ, model=None):
        self.imgsz = imgsz
        if model is None:
            model = build_mrcnn(imgsz); model.load_state_dict(torch.load(weights, map_location="cpu"))
        self.m = model.to(DEV).eval()
    def n_params(self): return sum(p.numel() for p in self.m.parameters())
    @torch.no_grad()
    def __call__(self, img, conf=CONF_EVAL, imgsz=None):
        s = imgsz or self.imgsz
        self.m.transform.min_size, self.m.transform.max_size = (s,), s
        x = TF.to_tensor(img[..., ::-1].copy()).to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            o = self.m([x])[0]
        k = o["scores"] >= conf
        return {"boxes": o["boxes"][k].float().cpu(), "scores": o["scores"][k].float().cpu(),
                "labels": (o["labels"][k] - 1).cpu(), "masks": (o["masks"][k, 0] > 0.5).cpu()}

def mask_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    A = torch.as_tensor(np.asarray(a)).to(DEV).flatten(1).float()
    B = torch.as_tensor(np.asarray(b)).to(DEV).flatten(1).float()
    inter = A @ B.T
    union = A.sum(1)[:, None] + B.sum(1)[None] - inter
    return (inter / union.clamp(min=1)).cpu().numpy()

def match(pred, gt_masks, gt_labels, conf=CONF_VIS, thr=0.5):
    # Pencocokan serakah per citra pada IoU mask ≥ thr untuk menghitung TP, FP, FN
    k = pred["scores"] >= conf
    pm, pl, ps = pred["masks"][k].numpy(), pred["labels"][k].numpy(), pred["scores"][k].numpy()
    iou = mask_iou(pm, gt_masks)
    used, ious, fp_cls, wrong = set(), [], [], 0
    for i in np.argsort(-ps):
        cand = [(iou[i, j], j) for j in range(len(gt_labels)) if j not in used and gt_labels[j] == pl[i] and iou[i, j] >= thr]
        if cand:
            v, j = max(cand); used.add(j); ious.append(v)
        else:
            fp_cls.append(NAMES[pl[i]])
            if len(gt_labels) and iou[i].max() >= thr and gt_labels[iou[i].argmax()] != pl[i]: wrong += 1
    fn_cls = [NAMES[gt_labels[j]] for j in range(len(gt_labels)) if j not in used]
    return {"GT": len(gt_labels), "TP": len(ious), "FP": len(fp_cls), "FN": len(fn_cls), "salah_kelas": wrong,
            "IoU_TP": round(float(np.mean(ious)), 3) if ious else np.nan,
            "FN_kelas": ", ".join(fn_cls), "FP_kelas": ", ".join(fp_cls)}

def keep_only(p, gm, gl, keep):
    # Hanya mempertahankan prediksi dan ground truth dari kelas di `keep`
    if keep is None: return p, gm, gl
    k, g = torch.as_tensor(np.isin(p["labels"].numpy(), keep)), np.isin(gl, keep)
    return {key: v[k] for key, v in p.items()}, gm[g], gl[g]

def evaluate(pred_fn, items, group_fn=None, imgsz=None, per_image=False, keep=None):
    metrics, counts, times, img_rows = {}, {}, [], []
    for n, (ip, lp) in enumerate(items):
        img, gm, gl = gt_of(ip, lp)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        p = pred_fn(img, imgsz=imgsz)
        torch.cuda.synchronize()
        if n >= 3: times.append((time.perf_counter() - t0) * 1000)
        p, gm, gl = keep_only(p, gm, gl, keep)
        gmt = torch.as_tensor(gm)
        gbox = masks_to_boxes(gmt) if len(gmt) else torch.zeros((0, 4))
        for g in {"semua", group_fn(ip) if group_fn else "semua"}:
            ms = metrics.setdefault(g, {"segm": MeanAveragePrecision(iou_type="segm", class_metrics=True, backend="pycocotools"),
                                        "bbox": MeanAveragePrecision(iou_type="bbox", backend="pycocotools")})
            ms["segm"].update([{"masks": p["masks"], "scores": p["scores"], "labels": p["labels"]}],
                              [{"masks": gmt, "labels": torch.as_tensor(gl)}])
            ms["bbox"].update([{"boxes": p["boxes"], "scores": p["scores"], "labels": p["labels"]}],
                              [{"boxes": gbox, "labels": torch.as_tensor(gl)}])
            counts[g] = counts.get(g, 0) + 1
        if per_image:
            img_rows.append({"img": ip.name, "grup": group_fn(ip) if group_fn else "semua", **match(p, gm, gl)})
    out = {}
    for g, ms in metrics.items():
        s, b = ms["segm"].compute(), ms["bbox"].compute()
        cls = np.atleast_1d(s["classes"].numpy()).tolist(); apc = np.atleast_1d(s["map_per_class"].numpy()).tolist()
        out[g] = {"n_citra": counts[g], "mask_mAP50_95": float(s["map"]), "mask_mAP50": float(s["map_50"]),
                  "mask_mAP75": float(s["map_75"]), "mask_AP_kecil": float(s["map_small"]),
                  "mask_AP_sedang": float(s["map_medium"]), "mask_AP_besar": float(s["map_large"]),
                  "box_mAP50_95": float(b["map"]), "box_mAP50": float(b["map_50"]),
                  "AP_per_kelas": {NAMES[int(c)]: float(v) for c, v in zip(cls, apc)}}
    return out, (float(np.mean(times)) if times else np.nan), pd.DataFrame(img_rows)

### 3.1 Training YOLO-seg (M1, M2, dan variasi ukuran input)

Semua run memakai bobot pretrained COCO, augmentasi bawaan Ultralytics, seed 42, dan anggaran waktu GPU yang sama (`TIME_H`). Ultralytics menyesuaikan jumlah epoch dan jadwal learning rate agar pas dengan anggaran waktu itu. Run yang bobotnya sudah ada, baik di folder kerja maupun di output versi sebelumnya, dilewati.

In [ ]:
def restore_run(name):
    if (RUNS/name/"weights"/"best.pt").exists(): return
    for p in PREV:
        if (p/"runs"/name/"weights"/"best.pt").exists():
            shutil.copytree(p/"runs"/name, RUNS/name, dirs_exist_ok=True); print(name, "dipulihkan dari", p); return

def train_yolo(name, weights, imgsz, batch):
    restore_run(name); best = RUNS/name/"weights"/"best.pt"
    if best.exists(): print(f"{name}: bobot sudah ada, training dilewati"); return best
    t0 = time.time()
    YOLO(weights).train(data=str(DATA_YAML), imgsz=imgsz, epochs=EPOCHS_MAX, time=TIME_H, batch=batch, seed=SEED,
                        project=str(RUNS), name=name, exist_ok=True, device=0, workers=4, patience=100, plots=True, verbose=False)
    log_gpu(name, time.time() - t0, model=weights, imgsz=imgsz, batch=batch)
    return best

EXP_YOLO = {
    "M1_yolov8n-seg_640": dict(weights="yolov8n-seg.pt", imgsz=640, batch=16),
    "M2_yolo11n-seg_640": dict(weights="yolo11n-seg.pt", imgsz=640, batch=16),
    "M2_yolo11n-seg_320": dict(weights="yolo11n-seg.pt", imgsz=320, batch=32),
    "M2_yolo11n-seg_960": dict(weights="yolo11n-seg.pt", imgsz=960, batch=8),
    # Variabel NRP untuk ketiga model: M1 juga dilatih ulang pada 320 dan 960 (ditambahkan di versi 4)
    "M1_yolov8n-seg_320": dict(weights="yolov8n-seg.pt", imgsz=320, batch=32),
    "M1_yolov8n-seg_960": dict(weights="yolov8n-seg.pt", imgsz=960, batch=8),
}
for name, cfg in EXP_YOLO.items():
    print("=" * 70, "\n", name, cfg)
    train_yolo(name, **cfg)
    torch.cuda.empty_cache()

### 3.2 Training Mask R-CNN (M3)

Mask R-CNN dari torchvision dilatih dengan SGD (momentum 0,9, weight decay 1e-4, learning rate 0,005, batch 4), warmup 500 iterasi, lalu learning rate turun mengikuti kosinus terhadap waktu yang terpakai sehingga anggarannya sama dengan YOLO. Augmentasinya hanya flip horizontal. Setiap akhir epoch model dievaluasi pada maksimal 300 citra validasi, bobot dengan mask mAP50-95 tertinggi disimpan, dan semua angka ditulis ke `results.csv`.

In [ ]:
def boxes_np(masks):
    b = np.zeros((len(masks), 4), np.float32)
    for k, m in enumerate(masks):
        ys, xs = np.nonzero(m); b[k] = [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]
    return b

class SegDataset(torch.utils.data.Dataset):
    def __init__(self, items, train): self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        img, masks, labels = gt_of(*self.items[i])
        if self.train and random.random() < 0.5:
            img, masks = img[:, ::-1], masks[:, :, ::-1]
        boxes = boxes_np(masks)
        return TF.to_tensor(img[..., ::-1].copy()), {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.as_tensor(labels + 1, dtype=torch.int64),
            "masks": torch.as_tensor(np.ascontiguousarray(masks), dtype=torch.uint8)}

def collate(batch): return tuple(zip(*batch))

def train_mrcnn(name="M3_maskrcnn-r50fpn_640", imgsz=IMGSZ, batch=4, lr=0.005, warmup=500):
    restore_run(name); d = RUNS/name; best = d/"weights"/"best.pt"
    if best.exists(): print(f"{name}: bobot sudah ada, training dilewati"); return best
    (d/"weights").mkdir(parents=True, exist_ok=True)
    dl = torch.utils.data.DataLoader(SegDataset(items_of(SRC, "train"), True), batch_size=batch, shuffle=True,
                                     num_workers=4, collate_fn=collate, drop_last=True)
    val_items = items_of(SRC, "valid"); random.Random(SEED).shuffle(val_items); val_items = val_items[:VAL_MAX]
    m = build_mrcnn(imgsz).to(DEV)
    opt = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=lr, momentum=0.9, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    budget, t0, it, best_map, rows = TIME_H * 3600, time.time(), 0, -1.0, []
    for ep in range(1, EPOCHS_MAX + 1):
        m.train(); sums, n = {}, 0
        for imgs, tgts in dl:
            el = time.time() - t0
            if el > budget: break
            f = min(1.0, (it + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(el / budget, 1.0)))
            for g in opt.param_groups: g["lr"] = lr * f
            imgs = [x.to(DEV) for x in imgs]; tgts = [{k: v.to(DEV) for k, v in t.items()} for t in tgts]
            with torch.autocast("cuda", dtype=torch.float16):
                ld = m(imgs, tgts)
            loss = sum(ld.values())
            opt.zero_grad()
            if not torch.isfinite(loss): print("loss tidak hingga, batch dilewati"); continue
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            for k, v in ld.items(): sums[k] = sums.get(k, 0.0) + v.item()
            n += 1; it += 1
        if n == 0: break
        res, _, _ = evaluate(MaskRCNNSeg(model=m, imgsz=imgsz), val_items)
        r = {"epoch": ep, "iterasi": it, "menit": round((time.time() - t0) / 60, 2), "lr": opt.param_groups[0]["lr"],
             **{f"train/{k}": v / n for k, v in sums.items()},
             "val/mask_mAP50_95": res["semua"]["mask_mAP50_95"], "val/mask_mAP50": res["semua"]["mask_mAP50"],
             "val/box_mAP50_95": res["semua"]["box_mAP50_95"]}
        rows.append(r); pd.DataFrame(rows).to_csv(d/"results.csv", index=False)
        print(f"epoch {ep:3d} | {r['menit']:6.1f} menit | loss {sum(v / n for v in sums.values()):.3f} | val mask mAP50-95 {r['val/mask_mAP50_95']:.4f}")
        torch.save(m.state_dict(), d/"weights"/"last.pt")
        if r["val/mask_mAP50_95"] > best_map: best_map = r["val/mask_mAP50_95"]; torch.save(m.state_dict(), best)
        if time.time() - t0 > budget: break
    log_gpu(name, time.time() - t0, model="maskrcnn_resnet50_fpn (COCO)", imgsz=imgsz, batch=batch)
    return best

train_mrcnn(); torch.cuda.empty_cache()
# Variabel NRP untuk Mask R-CNN: dilatih ulang pada 320 dan 960 (ditambahkan di versi 4).
# Batch dan learning rate 960 diperkecil bersamaan agar muat di memori T4
train_mrcnn("M3_maskrcnn-r50fpn_320", imgsz=320, batch=8, lr=0.01); torch.cuda.empty_cache()
train_mrcnn("M3_maskrcnn-r50fpn_960", imgsz=960, batch=2, lr=0.0025, warmup=1000); torch.cuda.empty_cache()

In [ ]:
# Kurva validasi selama training terhadap waktu. Angka YOLO dihitung oleh Ultralytics, angka Mask R-CNN oleh torchmetrics
def curve(name):
    f = RUNS/name/"results.csv"
    if not f.exists(): return None
    df = pd.read_csv(f); df.columns = [c.strip() for c in df.columns]
    if "menit" in df: return df["menit"], df["val/mask_mAP50_95"]
    x = df["time"] / 60 if "time" in df else df["epoch"]
    return x, df["metrics/mAP50-95(M)"]

fig, ax = plt.subplots(figsize=(8, 4.5))
for key, name, label in (("M1", "M1_yolov8n-seg_640", "M1 YOLOv8n-seg"), ("M2", "M2_yolo11n-seg_640", "M2 YOLO11n-seg"),
                         ("M3", "M3_maskrcnn-r50fpn_640", "M3 Mask R-CNN")):
    c = curve(name)
    if c is not None: ax.plot(*c, color=COLORS[key], lw=2, label=label)
ax.set_xlabel("waktu training (menit)"); ax.set_ylabel("mask mAP50-95 validasi")
ax.set_title("Kurva validasi selama training"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig(FIG/"kurva_validasi.png", dpi=150); plt.show()
display(pd.DataFrame(gpu_log).T)

## 4. Evaluasi pada test set asli dan test set sendiri

Tabel utama membandingkan M1, M2, dan M3 pada ukuran input 640. Baris tambahan menjawab variabel NRP, yaitu ketiga model yang dilatih ulang pada 320 dan 960, ditambah Mask R-CNN yang dilatih pada 640 lalu hanya diuji pada 320 dan 960 sebagai pembanding antara melatih ulang dan sekadar mengganti ukuran saat inferensi. Waktu inferensi diukur per citra dengan batch 1 pada GPU, termasuk pra-proses dan pasca-proses.

Foto sendiri hanya dianotasi untuk sebagian kelas, sehingga evaluasinya dibatasi pada kelas-kelas itu. Agar selisihnya adil, test set asli juga dievaluasi ulang dengan kelas yang sama (baris `asli_kelas_sendiri`), dan baris inilah yang dipakai sebagai pembanding domain shift.

Baris `valid_asli` menilai split valid dengan evaluator yang sama. Angkanya seharusnya dekat dengan mAP validasi yang dilaporkan saat training, sehingga selisih besar antara `valid_asli` dan `asli` menunjukkan bahwa split test memang lebih sulit, bukan bahwa evaluatornya keliru.

In [ ]:
def load_pred(run, imgsz):
    w = RUNS/run/"weights"/"best.pt"
    if not w.exists(): return None
    return MaskRCNNSeg(w, imgsz) if "maskrcnn" in run else YoloSeg(w, imgsz)

EVAL = [  # (label, run, ukuran input saat inferensi, utama?)
    ("M1 YOLOv8n-seg", "M1_yolov8n-seg_640", 640, True),
    ("M2 YOLO11n-seg", "M2_yolo11n-seg_640", 640, True),
    ("M3 Mask R-CNN R50-FPN", "M3_maskrcnn-r50fpn_640", 640, True),
    ("M2 YOLO11n-seg, latih 320", "M2_yolo11n-seg_320", 320, False),
    ("M2 YOLO11n-seg, latih 960", "M2_yolo11n-seg_960", 960, False),
    ("M1 YOLOv8n-seg, latih 320", "M1_yolov8n-seg_320", 320, False),
    ("M1 YOLOv8n-seg, latih 960", "M1_yolov8n-seg_960", 960, False),
    ("M3 Mask R-CNN, latih 320", "M3_maskrcnn-r50fpn_320", 320, False),
    ("M3 Mask R-CNN, latih 960", "M3_maskrcnn-r50fpn_960", 960, False),
    ("M3 Mask R-CNN, inferensi 320", "M3_maskrcnn-r50fpn_640", 320, False),
    ("M3 Mask R-CNN, inferensi 960", "M3_maskrcnn-r50fpn_640", 960, False),
]
rows, ap_rows, PER_IMAGE = [], [], {}
for label, run, s, main in EVAL:
    pred = load_pred(run, s)
    if pred is None: print("bobot belum ada:", run); continue
    plan = [("asli", TEST_ASLI, None, None)]
    # Cek silang: split valid dinilai dengan evaluator yang sama agar bisa dibandingkan dengan angka validasi saat training
    if main: plan += [("valid_asli", items_of(SRC, "valid"), None, None)]
    if TEST_OWN: plan += [("asli_kelas_sendiri", TEST_ASLI, None, OWN_KEEP), ("sendiri", TEST_OWN, own_group, OWN_KEEP)]
    for set_name, items, gf, keep in plan:
        res, ms, img_df = evaluate(pred, items, gf, imgsz=s, per_image=(main and set_name == "sendiri"), keep=keep)
        if main and set_name == "sendiri": PER_IMAGE[label] = img_df
        for g, r in res.items():
            test_set = set_name if g == "semua" else g
            rows.append({"model": label, "run": run, "imgsz": s, "utama": main, "test_set": test_set,
                         **{k: v for k, v in r.items() if k != "AP_per_kelas"},
                         "inferensi_ms": round(ms, 2), "params_M": round(pred.n_params() / 1e6, 2),
                         "jam_GPU_latih": gpu_log.get(run, {}).get("jam")})
            ap_rows += [{"model": label, "imgsz": s, "test_set": test_set, "kelas": c, "AP_mask": v} for c, v in r["AP_per_kelas"].items()]
        print(f"{label:32s} {set_name:8s} mask mAP50-95 {res['semua']['mask_mAP50_95']:.4f} | {ms:.1f} ms/citra")
    del pred; torch.cuda.empty_cache()

HASIL = pd.DataFrame(rows); HASIL.to_csv(OUT/"hasil_evaluasi.csv", index=False)
AP = pd.DataFrame(ap_rows); AP.to_csv(OUT/"ap_per_kelas.csv", index=False)
for k, v in PER_IMAGE.items(): v.to_csv(OUT/f"per_citra_sendiri_{k.split()[0]}.csv", index=False)
cols = ["model", "test_set", "n_citra", "mask_mAP50_95", "mask_mAP50", "mask_AP_kecil", "mask_AP_sedang", "mask_AP_besar",
        "box_mAP50_95", "box_mAP50", "inferensi_ms", "params_M", "jam_GPU_latih"]
display(HASIL[HASIL.utama][cols].round(4))

In [ ]:
# Domain shift: selisih performa antara test set asli dan test set sendiri
main = HASIL[HASIL.utama]
if TEST_OWN:
    pv = main.pivot_table(index="model", columns="test_set", values="mask_mAP50_95")
    # Pembanding yang adil adalah test set asli yang dibatasi pada kelas yang sama dengan foto sendiri
    BASE = "asli_kelas_sendiri"
    pv["selisih_sendiri_vs_asli"] = pv["sendiri"] - pv[BASE]
    pv["penurunan_relatif_%"] = 100 * pv["selisih_sendiri_vs_asli"] / pv[BASE]
    if "sendiri_sulit" in pv: pv["selisih_sulit_vs_normal"] = pv["sendiri_sulit"] - pv["sendiri_normal"]
    display(pv.round(4)); pv.to_csv(OUT/"domain_shift.csv")

    sets = [s for s in (BASE, "sendiri_normal", "sendiri_sulit") if s in pv]
    ramp = {BASE: "#9cc3ee", "sendiri_normal": "#4a8fe0", "sendiri_sulit": "#1d4f91"}
    nama = {BASE: "test set asli, kelas yang sama", "sendiri_normal": "foto sendiri, normal", "sendiri_sulit": "foto sendiri, sulit"}
    x, w = np.arange(len(pv)), 0.8 / len(sets)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for i, s in enumerate(sets):
        b = ax.bar(x + (i - (len(sets) - 1) / 2) * w, pv[s], w * 0.92, color=ramp[s], label=nama[s])
        ax.bar_label(b, fmt="%.3f", fontsize=8, padding=2)
    ax.set_xticks(x, pv.index); ax.set_ylabel("mask mAP50-95")
    ax.set_title("Performa turun saat diuji pada foto ruangan sendiri"); ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
    plt.tight_layout(); plt.savefig(FIG/"domain_shift.png", dpi=150); plt.show()

    best_label = pv["sendiri"].idxmax()
    a = AP[(AP.model == best_label)].pivot_table(index="kelas", columns="test_set", values="AP_mask")
    a = a[(a[BASE] >= 0) & (a["sendiri"] >= 0)]   # AP -1 berarti kelas itu tidak ada di ground truth set tersebut
    a["selisih"] = a["sendiri"] - a[BASE]; a = a.sort_values("selisih")
    display(a.round(3))
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(a))))
    y = np.arange(len(a))
    ax.barh(y + 0.2, a[BASE], 0.38, color="#9cc3ee", label="test set asli")
    ax.barh(y - 0.2, a["sendiri"], 0.38, color="#1d4f91", label="foto sendiri")
    ax.set_yticks(y, a.index); ax.set_xlabel("AP mask (IoU 0,50–0,95)")
    ax.set_title(f"AP per kelas, {best_label}"); ax.legend(frameon=False, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="x", alpha=0.25)
    plt.tight_layout(); plt.savefig(FIG/"ap_per_kelas.png", dpi=150); plt.show()

In [ ]:
# Variabel NRP: ukuran input 320 / 640 / 960 untuk ketiga model
SERI = {"M1_yolov8n-seg": ("M1 YOLOv8n-seg, dilatih pada ukuran itu", COLORS["M1"], "-"),
        "M2_yolo11n-seg": ("M2 YOLO11n-seg, dilatih pada ukuran itu", COLORS["M2"], "-"),
        "M3_maskrcnn-r50fpn": ("M3 Mask R-CNN, dilatih pada ukuran itu", COLORS["M3"], "-")}
SERI_INF = ("M3 Mask R-CNN, dilatih 640, hanya diuji pada ukuran itu", COLORS["M3"], "--")
rows_nrp = []
for _, r in HASIL[HASIL.test_set.isin(["asli", "sendiri", "sendiri_sulit"])].iterrows():
    pre, trained = r.run.rsplit("_", 1)
    if pre in SERI and int(trained) == r.imgsz: rows_nrp.append({**r, "seri": SERI[pre][0]})
    if pre == "M3_maskrcnn-r50fpn" and trained == "640": rows_nrp.append({**r, "seri": SERI_INF[0]})
nrp = pd.DataFrame(rows_nrp)
cols_nrp = ["seri", "imgsz", "test_set", "mask_mAP50_95", "mask_mAP50", "mask_AP_kecil", "mask_AP_sedang", "mask_AP_besar", "inferensi_ms", "jam_GPU_latih"]
NRP = nrp[cols_nrp].sort_values(["seri", "test_set", "imgsz"])
display(NRP.round(4)); NRP.to_csv(OUT/"eksperimen_ukuran_input.csv", index=False)
display(nrp.pivot_table(index="seri", columns=["test_set", "imgsz"], values="mask_mAP50_95").round(4))

panels = [("asli", "mask mAP50-95, test set asli"), ("sendiri", "mask mAP50-95, foto sendiri"), ("ms", "waktu inferensi (ms/citra)")]
panels = [p for p in panels if p[0] != "sendiri" or TEST_OWN]
fig, axs = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 4.4))
for ax, (key, title) in zip(np.atleast_1d(axs), panels):
    for nama, col, ls in list(SERI.values()) + [SERI_INF]:
        d = nrp[(nrp.seri == nama) & (nrp.test_set == ("asli" if key == "ms" else key))].sort_values("imgsz")
        if d.empty: continue
        yv = d["inferensi_ms"] if key == "ms" else d["mask_mAP50_95"]
        ax.plot(d["imgsz"], yv, ls, marker="o", color=col, lw=2, ms=7, label=nama, alpha=0.6 if ls == "--" else 1)
        for xx, yy in zip(d["imgsz"], yv):
            ax.annotate(f"{yy:.3f}" if key != "ms" else f"{yy:.0f}", (xx, yy), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7)
    ax.set_xticks(SIZES_NRP); ax.set_xlabel("ukuran input (piksel)"); ax.set_title(title, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25)
handles, labels = np.atleast_1d(axs)[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, fontsize=8)
plt.tight_layout(rect=(0, 0.12, 1, 1)); plt.savefig(FIG/"ukuran_input.png", dpi=150); plt.show()

## 5. Analisis kegagalan pada test set sendiri

Setiap citra di test set sendiri diberi skor kegagalan berupa jumlah FN dan FP dari ketiga model (ambang skor 0,25, IoU mask ≥ 0,5). Tabel ini membantu memilih 5 foto yang paling menarik untuk dianalisis. Isi `FAILURE_IMAGES` dengan nama file pilihanmu, atau biarkan kosong untuk memakai 5 teratas.

In [ ]:
FAILURE_IMAGES = ["sulit_09_oklusi", "sulit_23_redup", "sulit_24_redup", "normal_27", "normal_25"]   # awalan nama file; kosongkan untuk 5 teratas

PRED_CACHE = {}
def get_pred(label):
    run = {l: r for l, r, s, main in EVAL if main}[label]
    if label not in PRED_CACHE: PRED_CACHE[label] = load_pred(run, IMGSZ)
    return PRED_CACHE[label]
MAIN_LABELS = [l for l, r, s, main in EVAL if main and (RUNS/r/"weights"/"best.pt").exists()]

def show_compare(img_name, conf=CONF_VIS):
    ip, lp = next((p, l) for p, l in TEST_OWN if p.name == img_name)
    img, gm, gl = gt_of(ip, lp)
    fig, axs = plt.subplots(2, 2, figsize=(14, 11))
    axs[0, 0].imshow(overlay(img, gm, gl)); axs[0, 0].set_title(f"Ground truth ({len(gl)} objek)")
    rows = []
    for ax, label in zip(axs.flat[1:], MAIN_LABELS):
        p, _, _ = keep_only(get_pred(label)(img, conf=conf), gm, gl, OWN_KEEP)
        mt = match(p, gm, gl, conf); rows.append({"model": label, **mt})
        ax.imshow(overlay(img, p["masks"].numpy(), p["labels"].numpy(), p["scores"].numpy()))
        ax.set_title(f"{label}  TP {mt['TP']} | FP {mt['FP']} | FN {mt['FN']}")
    for ax in axs.flat: ax.axis("off")
    fig.suptitle(img_name, fontsize=10); plt.tight_layout()
    plt.savefig(FIG/f"gagal_{Path(img_name).stem[:40]}.png", dpi=110); plt.show()
    display(pd.DataFrame(rows)[["model", "GT", "TP", "FP", "FN", "salah_kelas", "IoU_TP", "FN_kelas", "FP_kelas"]])

if PER_IMAGE:
    df = pd.concat([v.assign(model=k) for k, v in PER_IMAGE.items()])
    KANDIDAT = (df.groupby(["img", "grup"]).agg(GT=("GT", "first"), TP=("TP", "sum"), FP=("FP", "sum"), FN=("FN", "sum"),
                                                 salah_kelas=("salah_kelas", "sum")).reset_index())
    KANDIDAT["skor_gagal"] = KANDIDAT.FN + KANDIDAT.FP
    KANDIDAT = KANDIDAT.sort_values("skor_gagal", ascending=False); KANDIDAT.to_csv(OUT/"kandidat_kegagalan.csv", index=False)
    display(KANDIDAT.head(15))
    _names = [p.name for p, _ in TEST_OWN]
    FAILURE_IMAGES = [next(n for n in _names if n.startswith(f)) for f in FAILURE_IMAGES if any(n.startswith(f) for n in _names)] or KANDIDAT.img.head(5).tolist()
    for name in FAILURE_IMAGES: show_compare(name)
else:
    print("Test set sendiri belum ada, analisis kegagalan dilewati")

### 5.1 Eksperimen kecil untuk menguji penjelasan

Setiap penjelasan kegagalan perlu diuji, bukan hanya diduga. Dua alat disiapkan di bawah ini.

**Batas resolusi mask (oracle).** Mask R-CNN memprediksi mask pada grid 28×28 di dalam setiap box, lalu memperbesarnya ke ukuran box. YOLO-seg menyusun mask dari prototype beresolusi 1/4 input, yaitu 160×160 untuk input 640. Mask ground truth diperkecil ke resolusi itu lalu diperbesar kembali, kemudian IoU-nya dengan mask asli dihitung. Nilai ini adalah batas atas kualitas mask yang bisa dicapai model **walaupun prediksinya sempurna**. Jika batas atasnya rendah untuk suatu objek, kegagalan mask (tepi bocor, bagian tipis hilang) berasal dari representasi mask, bukan dari kurangnya training.

**Uji varian input.** Foto yang gagal diprediksi ulang dengan satu perubahan saja, misalnya ukuran input diperbesar (menguji dugaan objek terlalu kecil) atau citra dicerahkan dengan gamma dan CLAHE (menguji dugaan pencahayaan). Jika TP naik setelah perubahan itu, dugaan tersebut didukung bukti.

In [ ]:
def oracle_mrcnn(m, grid=28):
    ys, xs = np.nonzero(m); x0, y0, x1, y1 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
    small = cv2.resize(m[y0:y1, x0:x1].astype(np.float32), (grid, grid), interpolation=cv2.INTER_AREA)
    back = np.zeros_like(m); back[y0:y1, x0:x1] = cv2.resize(small, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) > 0.5
    return (back & m).sum() / max((back | m).sum(), 1)

def oracle_yolo(m, imgsz=IMGSZ):
    H, W = m.shape; s = imgsz / max(H, W)
    small = cv2.resize(m.astype(np.float32), (max(1, round(W * s / 4)), max(1, round(H * s / 4))), interpolation=cv2.INTER_AREA)
    back = cv2.resize(small, (W, H), interpolation=cv2.INTER_LINEAR) > 0.5
    return (back & m).sum() / max((back | m).sum(), 1)

def oracle_table(items, tag, limit=200):
    rows = []
    for ip, lp in items[:limit]:
        img, gm, gl = gt_of(ip, lp)
        for m, c in zip(gm, gl):
            a = int(m.sum())
            rows.append({"set": tag, "img": ip.name, "kelas": NAMES[c], "luas": a, "ukuran": size_cat(a),
                         "oracle_MaskRCNN_28x28": oracle_mrcnn(m), "oracle_YOLO_proto": oracle_yolo(m)})
    return pd.DataFrame(rows)

ORA = pd.concat([oracle_table(TEST_ASLI, "asli")] + ([oracle_table(TEST_OWN, "sendiri")] if TEST_OWN else []))
ORA.to_csv(OUT/"oracle_resolusi_mask.csv", index=False)
print("Rata-rata IoU oracle per ukuran objek:")
display(ORA.groupby(["set", "ukuran"])[["oracle_MaskRCNN_28x28", "oracle_YOLO_proto"]].mean().round(3))
print("Kelas dengan IoU oracle Mask R-CNN terendah:")
display(ORA.groupby("kelas")[["oracle_MaskRCNN_28x28", "oracle_YOLO_proto"]].mean().sort_values("oracle_MaskRCNN_28x28").head(10).round(3))
if TEST_OWN and FAILURE_IMAGES:
    print("IoU oracle untuk objek di foto kegagalan:")
    display(ORA[ORA.img.isin(FAILURE_IMAGES)].sort_values(["img", "oracle_MaskRCNN_28x28"]).round(3))

In [ ]:
def gamma(img, g):
    lut = (np.linspace(0, 1, 256) ** g * 255).astype(np.uint8); return cv2.LUT(img, lut)
def clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

VARIAN = {  # nama: (fungsi citra, ukuran input)
    "asli 640": (lambda x: x, 640),
    "input 960": (lambda x: x, 960),
    "input 1280": (lambda x: x, 1280),
    "gamma 0,5 (lebih terang)": (lambda x: gamma(x, 0.5), 640),
    "CLAHE": (clahe, 640),
}

def test_variants(label, img_name, variants=VARIAN, conf=CONF_VIS):
    ip, lp = next((p, l) for p, l in TEST_OWN if p.name == img_name)
    img, gm, gl = gt_of(ip, lp); pred = get_pred(label); rows = []
    for vname, (fn, s) in variants.items():
        p, _, _ = keep_only(pred(fn(img), conf=conf, imgsz=s), gm, gl, OWN_KEEP)
        rows.append({"model": label, "img": img_name, "varian": vname, **match(p, gm, gl, conf)})
    return pd.DataFrame(rows)

if TEST_OWN and FAILURE_IMAGES:
    UJI = pd.concat([test_variants(l, n) for n in FAILURE_IMAGES for l in MAIN_LABELS])
    UJI.to_csv(OUT/"uji_varian_kegagalan.csv", index=False)
    display(UJI[["img", "model", "varian", "GT", "TP", "FP", "FN", "IoU_TP", "FN_kelas"]])

## 6. Bukti proses dan ekspor

In [ ]:
# EXIF foto asli. Unggah foto asli dari HP sebagai Kaggle Dataset privat bernama "foto-asli-cpmk3", lalu Add Input
try:
    from pillow_heif import register_heif_opener; register_heif_opener()
except Exception: pass
from PIL.ExifTags import TAGS
EXIF_DIR = ""   # kosongkan untuk mencari otomatis folder /kaggle/input yang namanya memuat "foto"
if not EXIF_DIR and IN_KAGGLE:
    cand = [d for d in glob.glob("/kaggle/input/**/*foto*", recursive=True) if os.path.isdir(d)]
    EXIF_DIR = cand[0] if cand else ""
if EXIF_DIR:
    rows = []
    for p in sorted(Path(EXIF_DIR).rglob("*")):
        if p.suffix.lower() not in (".jpg", ".jpeg", ".heic", ".png"): continue
        try: ex = Image.open(p).getexif()
        except Exception: continue
        d = {TAGS.get(k, k): v for k, v in ex.items()}
        d.update({TAGS.get(k, k): v for k, v in ex.get_ifd(0x8769).items()})
        rows.append({"file": p.name, "waktu_ambil": d.get("DateTimeOriginal", d.get("DateTime")), "merek": d.get("Make"),
                     "model_hp": d.get("Model"), "lebar": d.get("ExifImageWidth"), "tinggi": d.get("ExifImageHeight"),
                     "eksposur_s": str(d.get("ExposureTime")), "ISO": d.get("ISOSpeedRatings"), "f_number": str(d.get("FNumber"))})
    EXIF = pd.DataFrame(rows); EXIF.to_csv(OUT/"exif_foto_asli.csv", index=False)
    print(f"{len(EXIF)} foto dari {EXIF_DIR}"); display(EXIF)
else:
    print("Folder foto asli tidak ditemukan, tabel EXIF dilewati")

In [ ]:
WG = pd.DataFrame(gpu_log).T
display(WG); print(f"Total waktu GPU training: {WG['jam'].astype(float).sum():.2f} jam")

ringkas = {"versi": {"ultralytics": ultralytics.__version__, "torch": torch.__version__, "torchvision": torchvision.__version__,
                     "torchmetrics": torchmetrics.__version__, "gpu": GPU_NAME},
           "dataset": {"kelas": NAMES, **{s: len(items_of(SRC, s)) for s in ("train", "valid", "test")}, "sendiri": len(TEST_OWN)},
           "pengaturan": {"IMGSZ": IMGSZ, "SIZES_NRP": SIZES_NRP, "TIME_H": TIME_H, "SEED": SEED, "CONF_EVAL": CONF_EVAL, "CONF_VIS": CONF_VIS},
           "hasil": HASIL.drop(columns=["utama"]).to_dict("records"), "waktu_gpu": gpu_log, "foto_kegagalan": FAILURE_IMAGES}
(OUT/"ringkasan.json").write_text(json.dumps(ringkas, indent=2, default=str))

zp = WORK/"hasil_cpmk3.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and f.name != "last.pt": z.write(f, f.relative_to(WORK))
print("Selesai:", zp, f"{zp.stat().st_size / 1e6:.1f} MB")